# BankSim EDA

Exploratory analysis for *Fraudulent Detection in Banking System using Random Forest Classifier* (JETIR2405086).

Run this notebook from the project root so `config.py` resolves correctly, or keep the working directory at the repository root.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from utils.preprocessing import load_banksim, print_dataset_report, encode_age
from utils.feature_engineering import engineer_features

sns.set_theme(style="whitegrid")
df = load_banksim()
print_dataset_report(df)
df["age"] = encode_age(df["age"])
df.head()

## Class balance

Fraud is rare. That is why training uses class weights and keeps every fraud row.

In [ ]:
print(df["fraud"].value_counts())
print("Fraud rate:", df["fraud"].mean())
sns.countplot(data=df.sample(20000, random_state=42), x="fraud")
plt.title("Fraud vs genuine")
plt.show()

## Amount, age, gender, merchant category

In [ ]:
sample = df.sample(15000, random_state=42)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=sample, x="amount", hue="fraud", bins=40, ax=axes[0, 0])
axes[0, 0].set_xlim(0, sample["amount"].quantile(0.98))
sns.countplot(data=sample, x="age", hue="fraud", ax=axes[0, 1])
sns.countplot(data=sample, x="gender", hue="fraud", ax=axes[1, 0])
sns.countplot(data=sample, y="category", ax=axes[1, 1], order=sample["category"].value_counts().index)
plt.tight_layout()
plt.show()

## Feature engineering preview

In [ ]:
eng = engineer_features(df.sample(8000, random_state=1).copy())
# merchant risk needs fraud labels; sampling can distort rates — reload a stratified slice instead
fraud = df[df.fraud == 1].sample(min(2000, (df.fraud == 1).sum()), random_state=1)
ok = df[df.fraud == 0].sample(6000, random_state=1)
eng = engineer_features(pd.concat([fraud, ok]))
print(eng[["amount", "tx_frequency", "merchant_risk_score", "hour_of_tx", "tx_velocity", "spending_category"]].head())
sns.heatmap(eng[["amount", "tx_frequency", "avg_tx_amount", "merchant_risk_score", "tx_velocity", "fraud"]].corr(), annot=True, cmap="Blues")
plt.title("Correlation heatmap (engineered sample)")
plt.show()